# 04 — Skew Handling

Skewed joins, hot keys, partition diagnostics, AQE skew handling, and manual salting.

## 00 — Setup

```text
SparkSession
    │
    ├── warehouse: /tmp/spark-warehouse
    ├── shuffle partitions: 32
    ├── AQE toggled per section
    └── skew inspected through partition metrics
```

In [1]:
from pyspark.sql import SparkSession, functions as F
import time

spark = (
    SparkSession.builder
    .appName("skew_handling")
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "32")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark


## 01 — Balanced data

```text
events_balanced
  event_id
  user_id ─────────────┐
                       ├── join key
users                  │
  user_id ─────────────┘
  plan
  country

user_id distribution: mostly even
```

In [2]:
users = (
    spark.range(1, 200_001)
    .withColumnRenamed("id", "user_id")
    .withColumn("plan", F.expr("CASE WHEN user_id % 3 = 0 THEN 'pro' WHEN user_id % 3 = 1 THEN 'team' ELSE 'free' END"))
    .withColumn("country", F.expr("CASE WHEN user_id % 5 = 0 THEN 'SK' WHEN user_id % 5 = 1 THEN 'CZ' WHEN user_id % 5 = 2 THEN 'DE' WHEN user_id % 5 = 3 THEN 'AT' ELSE 'PL' END"))
)

events_balanced = (
    spark.range(0, 2_000_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("user_id", ((F.col("event_id") % 200_000) + 1).cast("long"))
    .withColumn("event_type", F.expr("CASE WHEN event_id % 4 = 0 THEN 'view' WHEN event_id % 4 = 1 THEN 'click' WHEN event_id % 4 = 2 THEN 'purchase' ELSE 'search' END"))
)

events_balanced.count(), users.count()


(2000000, 200000)

## 02 — Skewed data with a hot key

```text
normal events
  user_id = 2..200000
        │
        ├── union
        ▼
hot events
  user_id = 1 repeated many times

result: one key dominates the join
```

In [3]:
normal_events = (
    spark.range(0, 600_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("user_id", ((F.col("event_id") % 199_999) + 2).cast("long"))
    .withColumn("event_type", F.lit("normal"))
)

hot_events = (
    spark.range(0, 1_800_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("user_id", F.lit(1).cast("long"))
    .withColumn("event_type", F.lit("hot"))
)

events_skewed = normal_events.unionByName(hot_events)

events_skewed.groupBy("user_id").count().orderBy(F.desc("count")).show(10)


[Stage 6:>                                                          (0 + 2) / 4]

+-------+-------+
|user_id|  count|
+-------+-------+
|      1|1800000|
|      3|      4|
|      4|      4|
|      2|      4|
|     33|      3|
|    113|      3|
|     93|      3|
|    144|      3|
|    171|      3|
|    192|      3|
+-------+-------+
only showing top 10 rows


## 03 — Partition distribution before join

```text
repartition(user_id)
        │
        ▼
partition_id → row count
        │
        └── one partition much larger = skew symptom
```

In [4]:
partitioned_skewed = events_skewed.repartition(32, "user_id")

partition_distribution = (
    partitioned_skewed
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy(F.desc("count"))
)

partition_distribution.show(32, truncate=False)


+------------+-------+
|partition_id|count  |
+------------+-------+
|29          |1819002|
|11          |19284  |
|0           |19038  |
|13          |19038  |
|7           |18972  |
|2           |18966  |
|17          |18915  |
|1           |18906  |
|3           |18892  |
|16          |18864  |
|23          |18852  |
|26          |18852  |
|20          |18826  |
|12          |18783  |
|22          |18756  |
|6           |18747  |
|5           |18723  |
|4           |18717  |
|28          |18690  |
|14          |18684  |
|9           |18675  |
|10          |18627  |
|8           |18618  |
|19          |18618  |
|15          |18600  |
|30          |18594  |
|25          |18570  |
|24          |18553  |
|31          |18510  |
|27          |18501  |
|21          |18393  |
|18          |18234  |
+------------+-------+



## 04 — Balanced join baseline

```text
events_balanced ── shuffle(user_id) ──┐
                                      ├── join
users           ── shuffle(user_id) ──┘

reducers receive similar amounts of data
```

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

balanced_join = events_balanced.join(users, "user_id")
balanced_join.explain(mode="formatted")


== Physical Plan ==
* Project (11)
+- * SortMergeJoin Inner (10)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Project (3)
   :        +- * Filter (2)
   :           +- * Range (1)
   +- * Sort (9)
      +- Exchange (8)
         +- * Project (7)
            +- * Range (6)


(1) Range [codegen id : 1]
Output [1]: [id#4L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#4L]
Condition : isnotnull(((id#4L % 200000) + 1))

(3) Project [codegen id : 1]
Output [3]: [id#4L AS event_id#5L, ((id#4L % 200000) + 1) AS user_id#6L, CASE WHEN ((id#4L % 4) = 0) THEN view WHEN ((id#4L % 4) = 1) THEN click WHEN ((id#4L % 4) = 2) THEN purchase ELSE search END AS event_type#7]
Input [1]: [id#4L]

(4) Exchange
Input [3]: [event_id#5L, user_id#6L, event_type#7]
Arguments: hashpartitioning(user_id#6L, 32), ENSURE_REQUIREMENTS, [plan_id=300]

(5) Sort [codegen id : 2]
Input [3]: [event_id#5L, user_id#6L, event_type#7]
Arguments: [user_id#6L ASC NULLS FIR

## 05 — Skewed join without AQE

```text
events_skewed ── shuffle(user_id) ──┐
                                    ├── join
users         ── shuffle(user_id) ──┘

hot key goes to one reducer
that reducer becomes the straggler
```

In [6]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "32")

skewed_join_no_aqe = events_skewed.join(users, "user_id")
skewed_join_no_aqe.explain(mode="formatted")


== Physical Plan ==
* Project (14)
+- * SortMergeJoin Inner (13)
   :- * Sort (8)
   :  +- Exchange (7)
   :     +- Union (6)
   :        :- * Project (3)
   :        :  +- * Filter (2)
   :        :     +- * Range (1)
   :        +- * Project (5)
   :           +- * Range (4)
   +- * Sort (12)
      +- Exchange (11)
         +- * Project (10)
            +- * Range (9)


(1) Range [codegen id : 1]
Output [1]: [id#22L]
Arguments: Range (0, 600000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#22L]
Condition : isnotnull(((id#22L % 199999) + 2))

(3) Project [codegen id : 1]
Output [3]: [id#22L AS event_id#23L, ((id#22L % 199999) + 2) AS user_id#24L, normal AS event_type#25]
Input [1]: [id#22L]

(4) Range [codegen id : 2]
Output [1]: [id#26L]
Arguments: Range (0, 1800000, step=1, splits=Some(2))

(5) Project [codegen id : 2]
Output [3]: [id#26L AS event_id#27L, 1 AS user_id#28L, hot AS event_type#29]
Input [1]: [id#26L]

(6) Union

(7) Exchange
Input [3]: [event_id#

## 06 — Timing helper

```text
DataFrame
   │
   ├── action: count()
   └── elapsed seconds
```

In [7]:
def measure(label, df):
    start = time.time()
    rows = df.count()
    seconds = round(time.time() - start, 3)
    return (label, rows, seconds)

baseline_results = [
    measure("balanced_join", balanced_join),
    measure("skewed_join_no_aqe", skewed_join_no_aqe),
]

spark.createDataFrame(baseline_results, ["case", "rows", "seconds"]).show(truncate=False)


[Stage 23:>                                                         (0 + 1) / 1]

+------------------+-------+-------+
|case              |rows   |seconds|
+------------------+-------+-------+
|balanced_join     |2000000|1.33   |
|skewed_join_no_aqe|2400000|1.172  |
+------------------+-------+-------+



## 07 — AQE skew join handling

```text
AQE observes shuffle partition sizes
        │
        ├── normal partitions unchanged
        └── skewed partition split into smaller tasks

hot key work is divided
```

In [8]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1024")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

skewed_join_aqe = events_skewed.join(users, "user_id")
skewed_join_aqe.count()
skewed_join_aqe.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (15)
+- Project (14)
   +- SortMergeJoin Inner (13)
      :- Sort (8)
      :  +- Exchange (7)
      :     +- Union (6)
      :        :- Project (3)
      :        :  +- Filter (2)
      :        :     +- Range (1)
      :        +- Project (5)
      :           +- Range (4)
      +- Sort (12)
         +- Exchange (11)
            +- Project (10)
               +- Range (9)


(1) Range
Output [1]: [id#22L]
Arguments: Range (0, 600000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#22L]
Condition : isnotnull(((id#22L % 199999) + 2))

(3) Project
Output [3]: [id#22L AS event_id#23L, ((id#22L % 199999) + 2) AS user_id#24L, normal AS event_type#25]
Input [1]: [id#22L]

(4) Range
Output [1]: [id#26L]
Arguments: Range (0, 1800000, step=1, splits=Some(2))

(5) Project
Output [3]: [id#26L AS event_id#27L, 1 AS user_id#28L, hot AS event_type#29]
Input [1]: [id#26L]

(6) Union

(7) Exchange
Input [3]: [event_id#23L, user_id#24L, event_type#25]
Arguments

## 08 — Manual salting model

```text
events_skewed
  hot user_id = 1
        │
        ├── add random salt 0..N-1
        ▼
users
  hot user_id = 1
        │
        ├── duplicate row for salt 0..N-1
        ▼
join on (user_id, salt)

hot key is spread across N salted keys
```

In [9]:
SALT_BUCKETS = 8

salted_events = events_skewed.withColumn(
    "salt",
    F.when(F.col("user_id") == 1, (F.rand(seed=42) * SALT_BUCKETS).cast("int")).otherwise(F.lit(0))
)

salt_array = F.when(
    F.col("user_id") == 1,
    F.array(*[F.lit(i) for i in range(SALT_BUCKETS)])
).otherwise(F.array(F.lit(0)))

salted_users = users.withColumn("salt", F.explode(salt_array))

salted_events.groupBy("user_id", "salt").count().filter(F.col("user_id") == 1).orderBy("salt").show()


+-------+----+------+
|user_id|salt| count|
+-------+----+------+
|      1|   0|225209|
|      1|   1|225494|
|      1|   2|225171|
|      1|   3|224494|
|      1|   4|225217|
|      1|   5|224077|
|      1|   6|224919|
|      1|   7|225419|
+-------+----+------+



## 09 — Salted join

```text
salted_events ── shuffle(user_id, salt) ──┐
                                          ├── join
salted_users  ── shuffle(user_id, salt) ──┘

one hot key becomes multiple smaller join groups
```

In [10]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

salted_join = salted_events.join(salted_users, on=["user_id", "salt"], how="inner")
salted_join.explain(mode="formatted")


== Physical Plan ==
* Project (16)
+- * SortMergeJoin Inner (15)
   :- * Sort (9)
   :  +- Exchange (8)
   :     +- * Filter (7)
   :        +- * Project (6)
   :           +- Union (5)
   :              :- * Project (2)
   :              :  +- * Range (1)
   :              +- * Project (4)
   :                 +- * Range (3)
   +- * Sort (14)
      +- Exchange (13)
         +- * Generate (12)
            +- * Project (11)
               +- * Range (10)


(1) Range [codegen id : 1]
Output [1]: [id#22L]
Arguments: Range (0, 600000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [3]: [id#22L AS event_id#23L, ((id#22L % 199999) + 2) AS user_id#24L, normal AS event_type#25]
Input [1]: [id#22L]

(3) Range [codegen id : 2]
Output [1]: [id#26L]
Arguments: Range (0, 1800000, step=1, splits=Some(2))

(4) Project [codegen id : 2]
Output [3]: [id#26L AS event_id#27L, 1 AS user_id#28L, hot AS event_type#29]
Input [1]: [id#26L]

(5) Union

(6) Project [codegen id : 3]
Output [4]: [eve

## 10 — Salted partition distribution

```text
before salting
  hot key → one large partition

after salting
  hot key + salt → multiple smaller partitions
```

In [11]:
salted_distribution = (
    salted_events
    .repartition(32, "user_id", "salt")
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy(F.desc("count"))
)

salted_distribution.show(32, truncate=False)


+------------+------+
|partition_id|count |
+------------+------+
|17          |468417|
|25          |244187|
|31          |244064|
|29          |243950|
|13          |243594|
|6           |243579|
|3           |242947|
|9           |19461 |
|16          |19140 |
|11          |19095 |
|4           |19069 |
|10          |19023 |
|15          |19011 |
|23          |19008 |
|14          |18945 |
|8           |18927 |
|21          |18897 |
|1           |18897 |
|2           |18801 |
|22          |18777 |
|5           |18769 |
|7           |18744 |
|0           |18741 |
|18          |18648 |
|26          |18480 |
|19          |18474 |
|30          |18468 |
|27          |18426 |
|28          |18402 |
|12          |18399 |
|20          |18366 |
|24          |18294 |
+------------+------+



## 11 — Strategy comparison

```text
same skewed join problem
        │
        ├── no AQE       → straggler risk
        ├── AQE skew     → automatic runtime split
        └── manual salt  → deterministic key spreading
```

In [12]:
def timed_case(label, aqe_enabled, df):
    spark.conf.set("spark.sql.adaptive.enabled", "true" if aqe_enabled else "false")
    start = time.time()
    rows = df.count()
    return (label, rows, round(time.time() - start, 3))

comparison = [
    timed_case("skewed_no_aqe", False, skewed_join_no_aqe),
    timed_case("skewed_with_aqe", True, skewed_join_aqe),
    timed_case("manual_salting", False, salted_join),
]

spark.createDataFrame(comparison, ["case", "rows", "seconds"]).show(truncate=False)


+---------------+-------+-------+
|case           |rows   |seconds|
+---------------+-------+-------+
|skewed_no_aqe  |2400000|0.961  |
|skewed_with_aqe|2400000|0.795  |
|manual_salting |2400000|1.187  |
+---------------+-------+-------+



## 12 — Decision table

```text
skew detected
    │
    ├── dimension small enough → broadcast
    ├── AQE enabled            → try AQE skew handling
    ├── one known hot key      → manual salting
    └── many hot keys          → repartitioning / data model review
```

In [13]:
decision_rows = [
    ("small dimension", "broadcast", "Avoids shuffle on large side"),
    ("unknown skew at runtime", "AQE skew handling", "Spark splits skewed shuffle partitions"),
    ("known hot key", "manual salting", "Spread one key across multiple join keys"),
    ("many hot keys", "key design / repartitioning", "Fix data distribution instead of patching one key"),
    ("aggregate after join", "salt then unsalt aggregation", "Keep correctness after key spreading"),
]

spark.createDataFrame(decision_rows, ["case", "strategy", "reason"]).show(truncate=False)


+-----------------------+----------------------------+-------------------------------------------------+
|case                   |strategy                    |reason                                           |
+-----------------------+----------------------------+-------------------------------------------------+
|small dimension        |broadcast                   |Avoids shuffle on large side                     |
|unknown skew at runtime|AQE skew handling           |Spark splits skewed shuffle partitions           |
|known hot key          |manual salting              |Spread one key across multiple join keys         |
|many hot keys          |key design / repartitioning |Fix data distribution instead of patching one key|
|aggregate after join   |salt then unsalt aggregation|Keep correctness after key spreading             |
+-----------------------+----------------------------+-------------------------------------------------+

